# 02 · LLM: fine-tune Qwen3.5-4B on tool calling (ver2: LFM3 ver3's changes)

Trains a LoRA on the gold trajectories, merges it into the text-only model, and saves `lifemate.py` next to it, so inference always uses the prompt format these weights were trained on.

Built from `02_llm_qwen.ipynb`. Changes, all proven first on the 2-hour LFM3 benchmark (`02c_llm_lfm3_ver3.ipynb`), each measurable with `03c_eval_fixes.ipynb` and `EVAL = True`:
1. **History calls are training targets too** (`HISTORY_CALLS`): 3,661 gold calls from earlier turns on top of the 3,541 final ones.
2. **Clean text in transcript form**: the clean half goes through the same lowercasing, punctuation removal and ITN as transcripts. ASR text is required, marks are removed, `vi_itn.py` comes from the attached 01 output.
3. **Optional written reasons** (`WHY`, off by default): attach `rationales.json`, made locally by `rationales/gen_rationales.py` with no API.
4. **App masking** (`MASK_RATE`, 0.3): in that share of conversations every installed app gets a made-up name in the tool list, the calls, the `x-ref`s and the reasons, so the model has to read schemas instead of remembering app names. 65% of public samples use an app training never saw. Function masking as in Hammer (arXiv 2410.04587). Training only, inference is unchanged. 0 turns it off.
5. **Spark ver7's prompt**: fuller rules, `hôm nay` / `ngày mai` / `ngày kia` in the date table, `kind`, `returns` and `x-ref` in the tool text, the stricter scorer. The context stays in the system prompt.
6. Training logs host RAM, stops with a checkpoint under `MIN_FREE_GB`, and writes `train.log`.
The prompt changed, so do not attach a `ckpt/` or `llm_lora/` from an earlier version. Evaluate with 03c, not 03, and trust its `asr` scores.

**Setup**
1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add Input: the private dataset. and the output of `01c_asr_gipformer_ft.ipynb` with `FINAL = False` (01b also works). Attach the same output to 03c.

**Run**
1. Run all with `SMOKE = True`. It trains on the longest conversations, so out-of-memory and NaN problems appear in minutes. Note the seconds per step and the peak GB per GPU it prints.
2. Set `SMOKE = False`, then Save Version > Save & Run All. `MAX_HOURS` stops training and still saves before Kaggle's 12 hour limit.
3. After validating with `03c_eval_fixes.ipynb`, run again with `FINAL = True` for the submission.

**If the smoke run fails**: on OOM lower `DELTA_PART`, then `MAX_LEN`. On a non-finite loss read the error: it names the first bad module and any bad weights. If fp16 itself is at fault, switch `MODEL_ID` to the fallback, nothing else changes.

**If a full run dies**: Add Input the output of the failed version, set `RESUME = True`, and run again with the same settings. It continues from the newest checkpoint in `ckpt/`, saved every `SAVE_STEPS` updates.

**Output**: `llm_model/`, `lifemate.py`.

In [ ]:
import torch
assert torch.cuda.device_count() == 2, "Set Settings > Accelerator to GPU T4 x2"
print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])
!pip install -q "transformers==5.22.1" "peft==0.21.2" "accelerate==1.15.0"
!pip uninstall -q -y torchao  # unused, and peft refuses the old version Kaggle ships

In [ ]:
%%writefile lifemate.py
"""LifeMate shared code: data split, prompt format, action parser, scorer.
02 trains on this file and saves it next to the model; 03 imports that copy, so the prompt always matches the weights.
Self-check: python lifemate.py <data root>"""
import collections
import json
import random
import re
import sys
import unicodedata
from datetime import datetime, timedelta
from pathlib import Path

ASK = "sys.ask_missing_info"
BOS, STOP = "", "<|im_end|>"
ROLES = {r: f"<|im_start|>{r}\n" for r in ("system", "user", "assistant")}  # plain ChatML: Qwen templates add a think block only to the last assistant turn, which would make training and inference differ
GEN = ROLES["assistant"]
MAX_RESULT_CHARS = 1500  # ponytail: hard cut on long tool results, summarise per tool if truncation ever hides an ID
HOLDOUT_APPS = {"door_lock", "learning", "library", "loyalty", "subscriptions", "household_chores", "home_security"}  # train-only apps: stands in for the unseen apps of the test splits
RELATIVE = {0: " (hôm nay)", 1: " (ngày mai)", 2: " (ngày kia)"}
WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]
RULES = """Bạn là trợ lý giọng nói tiếng Việt. Hãy xử lý yêu cầu CUỐI CÙNG của người dùng bằng cách gọi tool, người dùng có thể đổi ý giữa chừng.
Mỗi lượt trả về đúng một dòng JSON, không viết gì khác:
- Gọi tool: {"name": "<app.tool>", "arguments": {...}}
- Kết thúc: {"done": true}
Quy tắc:
- Chỉ gọi tool trong danh sách, đúng tên và đúng kiểu tham số. Không tự đoán ID, hãy đọc bằng tool trước khi ghi. ID đã có trong hội thoại thì dùng lại, không đọc lại.
- Chỉ gọi tool thật sự cần. Đọc xong mà yêu cầu cần ghi thì phải ghi tiếp, đừng dừng ở bước đọc.
- Tham số không bắt buộc: chỉ điền khi người dùng nói rõ, không tự điền giá trị mặc định. Tool đọc không thêm bộ lọc nếu người dùng không nêu.
- Yêu cầu có điều kiện ("nếu mai mưa thì..."): đọc điều kiện trước, chỉ làm nhánh đúng. Điều kiện sai thì {"done": true}.
- Tìm thấy đúng một kết quả phù hợp thì dùng luôn. Không tìm thấy, nhiều kết quả không phân biệt được, hoặc thiếu tham số bắt buộc: làm xong các việc khác rồi gọi sys.ask_missing_info với tool_name là tool định gọi và params là tên các tham số còn thiếu, sau đó kết thúc.
- Không app nào làm được, người dùng tự huỷ ("à thôi khỏi"), phủ định ("đừng tắt"), hoặc chỉ trò chuyện: {"done": true} ngay, không gọi tool.
- Thời gian dùng ISO-8601 đủ giây và múi giờ, ví dụ 2026-11-11T05:30:00+07:00. Tra ngày trong bảng 14 ngày tới, không tự tính.
- Số tiền là số nguyên đồng (2 triệu = 2000000). Số đọc từng chữ số thì chép đúng thứ tự, số điện thoại giữ dạng chuỗi.
- Văn bản tự do (tin nhắn, tiêu đề, ghi chú): viết lại ngắn gọn ý chính, giữ dấu tiếng Việt, tên riêng và con số. Từ khoá tìm kiếm chỉ là tên hoặc cụm từ người dùng nhắc."""


def load_split(root, split):
    with open(Path(root) / split / f"{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def load_catalog(root, split):
    with open(Path(root) / split / f"{split}_tools_catalog.json", encoding="utf-8") as f:
        return {t["name"]: t for a in json.load(f)["apps"] for t in a["tools"]}


def used_apps(x):
    return {(c["arguments"]["tool_name"] if c["name"] == ASK else c["name"]).split(".")[0] for c in x["label"]["tool_calls"]}


def clips(x):
    """Audio paths of a sample: the final request plus every user turn of the history."""
    return [x["audio"]] + [t["audio"] for t in x["history"] if t["role"] == "user"]


def gold_texts(x):
    return {x["audio"]: x["audio_text"]} | {t["audio"]: t["audio_text"] for t in x["history"] if t["role"] == "user"}


def split_rows(rows):
    """(train, val_unseen_apps, val_seen_apps). Final run: train on all rows instead."""
    train, unseen, seen = [], [], []
    for i, x in enumerate(rows):
        (unseen if used_apps(x) & HOLDOUT_APPS else seen if i % 12 == 0 else train).append(x)
    return train, unseen, seen


def render_tool(t):
    """Keeps kind, returns and x-ref: they say what is a read and which read supplies an ID, which an unseen app needs."""
    p = t["parameters"]
    o = {"name": t["name"], "kind": t["kind"], "description": t["description"]}
    if p.get("properties"):
        o["parameters"] = {k: {a: b for a, b in v.items() if a != "x-match"} for k, v in p["properties"].items()}
    for k in ("required", "x-one-of"):
        if p.get(k):
            o[k] = p[k]
    if t.get("returns"):
        o["returns"] = t["returns"]
    return json.dumps(o, ensure_ascii=False, separators=(",", ":"))


def system_prompt(ctx, tools):
    """Context stays in the system prompt, not beside the request as in Spark ver3: history calls are training targets too, and they need the clock."""
    now = datetime.fromisoformat(ctx["current_time"])
    days = ", ".join(f"{WEEKDAYS[d.weekday()]} {d:%Y-%m-%d}{RELATIVE.get(i, '')}" for i, d in enumerate(now + timedelta(i) for i in range(14)))
    return (f"{RULES}\n\nBây giờ: {ctx['current_time']} ({WEEKDAYS[now.weekday()]})\n14 ngày tới: {days}\n"
            f"Vị trí: {json.dumps(ctx.get('location'), ensure_ascii=False)}\n"
            f"Người dùng: {json.dumps(ctx.get('user_profile'), ensure_ascii=False)}\n\nDanh sách tool:\n" + "\n".join(render_tool(t) for t in tools))


def asst(obj):
    return {"role": "assistant", "content": json.dumps(obj, ensure_ascii=False)}


def result_msg(result):
    return {"role": "user", "content": "KẾT QUẢ: " + json.dumps(result, ensure_ascii=False)[:MAX_RESULT_CHARS]}


def build_messages(item, texts, catalog, rng=None):
    """texts: {audio path: transcript}. rng shuffles the tools, so the model cannot learn a position. Roles strictly alternate."""
    ctx = item["context"]
    names = [t for a in ctx["installed_apps"].values() for t in a["tools"]] + [ASK]
    if rng:
        rng.shuffle(names)
    msgs = [{"role": "system", "content": system_prompt(ctx, [catalog[n] for n in names])}]
    for turn in item["history"]:
        if turn["role"] == "user":
            msgs.append({"role": "user", "content": texts[turn["audio"]]})
            continue
        results = turn.get("tool_results") or []
        for i, c in enumerate(turn.get("tool_calls") or []):
            msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(results[i] if i < len(results) else {"ok": True})]
        msgs.append(asst({"done": True, "text": turn.get("text", "")}))
    msgs.append({"role": "user", "content": texts[item["audio"]]})
    return msgs


def gold_steps(x):
    """Gold trajectory after the final user turn: call, result, ..., done."""
    msgs = []
    for c, r in zip(x["label"]["tool_calls"], x["label"]["tool_results"]):
        msgs += [asst({"name": c["name"], "arguments": c["arguments"]}), result_msg(r)]
    return msgs + [asst({"done": True})]


def parse_action(text):
    """(name, arguments), or None for done / unparsable output (both end the turn)."""
    try:
        o = json.JSONDecoder().raw_decode(text[text.index("{"):])[0]
    except ValueError:
        return None
    if not isinstance(o.get("name"), str):
        return None
    return o["name"], o["arguments"] if isinstance(o.get("arguments"), dict) else {}


def tokens(s):
    return re.findall(r"[^\W_]+", unicodedata.normalize("NFC", str(s).lower()))  # word tokens, so "nhé." equals "nhé". The official tokeniser is unconfirmed


def token_f1(a, b):
    a, b = tokens(a), tokens(b)
    c = sum((collections.Counter(a) & collections.Counter(b)).values())
    return 2 * c / (len(a) + len(b)) if c else 0.0


def same(a, b):
    return json.dumps(a, sort_keys=True) == json.dumps(b, sort_keys=True)  # type-strict: 1, 1.0 and true all differ


def call_match(p, g, catalog):
    pa, ga = p.get("arguments") or {}, g.get("arguments") or {}
    if p["name"] != g["name"] or set(pa) != set(ga):
        return False
    props = catalog.get(g["name"], {}).get("parameters", {}).get("properties", {})
    return all(token_f1(pa[k], v) > 0.6 if props.get(k, {}).get("x-match") == "fuzzy" else same(pa[k], v) for k, v in ga.items())


def matches(pred, gold, catalog):
    """Size of the maximum one-to-one matching (Kuhn), which a greedy pass can undercount."""
    edges = [[j for j, g in enumerate(gold) if call_match(p, g, catalog)] for p in pred]
    owner = {}

    def augment(i, seen):
        for j in edges[i]:
            if j not in seen:
                seen.add(j)
                if j not in owner or augment(owner[j], seen):
                    owner[j] = i
                    return True
        return False

    return sum(augment(i, set()) for i in range(len(pred)))


def score(pred, gold, pred_hash, gold_hash, catalog):
    """Official formula from the description."""
    if not gold:
        return float(not pred)
    if pred_hash != gold_hash:
        return 0.0
    m = matches(pred, gold, catalog)
    if not m:
        return 0.0
    precision, recall = m / len(pred), m / len(gold)
    return 2 * precision * recall / (precision + recall)


def render(msgs):
    return "".join(f"{ROLES[m['role']]}{m['content']}{STOP}\n" for m in msgs)


def prompt_text(msgs):
    """Inference prompt, rendered here so that training and inference always see the same text."""
    return BOS + render(msgs) + GEN


def targets(msgs, first_target, history_calls):
    """Indices of the assistant turns to learn: every turn from first_target, plus the tool calls of history turns if history_calls.
    History replies ({"done": true, "text": ...}) stay unlabelled, since the final turn never writes text."""
    return {i for i, m in enumerate(msgs) if m["role"] == "assistant" and (i >= first_target or history_calls and m["content"].startswith('{"name"'))}


def mask_apps(msgs, apps, rng):
    """Training only: every installed app gets a made-up name in every turn, so the model has to read the tool list instead of
    remembering app names (function masking, as in Hammer, arXiv 2410.04587). Train results and transcripts never hold `app.`, checked."""
    new = {a: "".join(rng.choices("abcdefghijklmnopqrstuvwxyz", k=rng.randint(4, 9))) for a in apps}
    pat = re.compile(rf"(?<![\w@.])({'|'.join(map(re.escape, new))})\.(?=[a-z_])")
    return [m | {"content": pat.sub(lambda g: new[g.group(1)] + ".", m["content"])} for m in msgs]


def encode(tok, msgs, labelled):
    """Tokenise one conversation, labelling only the assistant turns whose index is in labelled."""
    text, spans = BOS, []
    for i, m in enumerate(msgs):
        start = len(text) + len(ROLES[m["role"]])
        text += render([m])
        if i in labelled:
            spans.append((start, len(text) - 1))  # content + end token, without the newline
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
    labels = [t if any(a <= s and e <= b for a, b in spans) else -100 for t, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}


def demo(root):
    sys.path.insert(0, str(Path(root) / "train"))
    from zaic_tools import ToolEnv

    rows, catalog = load_split(root, "train"), load_catalog(root, "train")
    for x in rows[:200]:  # gold replay must score full marks, otherwise the scorer or the data path is broken
        env = ToolEnv("train", x["id"], root=root)
        gold = x["label"]["tool_calls"]
        results = [env.call(c["name"], c["arguments"]) for c in gold]
        assert results == x["label"]["tool_results"], x["id"]
        assert score(env.get_tool_calls(), gold, "h", "h", catalog) == 1.0, x["id"]

    g = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "mai họp lúc chín giờ"}}]
    near = [{"name": "sms.send", "arguments": {"contact_id": "c_1", "content": "Mai họp lúc chín giờ nhé"}}]
    extra = g + [{"name": "contacts.search", "arguments": {"query": "x"}}]
    wrong_id = [{"name": "sms.send", "arguments": {"contact_id": "c_2", "content": "mai họp lúc chín giờ"}}]
    assert score(near, g, "a", "a", catalog) == 1.0
    assert score(g, g, "a", "b", catalog) == 0.0
    assert score(wrong_id, g, "a", "a", catalog) == 0.0
    assert score([], g, "a", "a", catalog) == 0.0
    assert score(g, [], "a", "a", catalog) == 0.0
    assert score([], [], "x", "y", catalog) == 1.0
    assert abs(score(extra, g, "a", "a", catalog) - 2 / 3) < 1e-9
    assert token_f1("mai họp nhé.", "Mai họp, nhé") == 1.0
    fz = {"t.x": {"parameters": {"properties": {"q": {"x-match": "fuzzy"}}}}}
    assert not call_match({"name": "t.x", "arguments": {"q": "a b", "n": 1}}, {"name": "t.x", "arguments": {"q": "a b"}}, fz)  # extra key
    assert not call_match({"name": "t.x", "arguments": {"n": 1}}, {"name": "t.x", "arguments": {"n": True}}, fz)  # type
    q = [{"name": "t.x", "arguments": {"q": s}} for s in ("một hai ba bốn", "hai ba năm sáu", "một hai ba năm")]
    assert matches(q[:2], [q[2], q[0]], fz) == 2  # the first pred fits both golds, the second only the first: greedy finds 1

    assert parse_action('{"name": "a.b", "arguments": {"x": 1}} rác') == ("a.b", {"x": 1})
    assert parse_action('rác {"name": "a.b"}') == ("a.b", {})
    assert parse_action('{"done": true}') is None
    assert parse_action("không có json") is None
    assert parse_action('{"name": ') is None

    train, unseen, seen = split_rows(rows)
    assert len(train) + len(unseen) + len(seen) == len(rows)
    assert not any(used_apps(x) & HOLDOUT_APPS for x in train)
    for x in rows:
        msgs = build_messages(x, gold_texts(x), catalog) + gold_steps(x)
        roles = [m["role"] for m in msgs[1:]]
        assert all(a != b for a, b in zip(roles, roles[1:])), x["id"]
        first = len(msgs) - 2 * len(x["label"]["tool_calls"]) - 1
        assert msgs[first - 1]["content"] == x["audio_text"], x["id"]
        hist = [c for t in x["history"] for c in t.get("tool_calls") or []]
        assert len(targets(msgs, first, True)) == len(hist) + len(x["label"]["tool_calls"]) + 1 and len(targets(msgs, first, False)) == len(x["label"]["tool_calls"]) + 1, x["id"]
        m = mask_apps(msgs, x["context"]["installed_apps"], random.Random(0))
        called = [json.loads(t["content"])["name"] for t in m[first:] if t["role"] == "assistant" and t["content"].startswith('{"name"')]
        assert all(f'"name":"{n}"' in m[0]["content"] for n in called) and not any(f'"name":"{a}.' in m[0]["content"] for a in x["context"]["installed_apps"]), x["id"]
    print("lifemate self-check ok:", len(train), "train,", len(unseen), "unseen-app val,", len(seen), "seen-app val")


if __name__ == "__main__":
    demo(sys.argv[1])

In [ ]:
%%writefile settings.py

import glob
from pathlib import Path

SMOKE = True  # True: a few steps on the longest conversations. Then False and Save & Run All.
SMOKE_STEPS = 20
FINAL = False  # True: train on all 1,545 samples, nothing held out
MODEL_ID = "Qwen/Qwen3.5-4B"  # fallback if fp16 goes non-finite on T4: "Qwen/Qwen3-4B-Instruct-2507"
DTYPE = "float16"
FP16 = True  # T4 has no bf16
MAX_LEN = 9000  # measured maximum 8,708 tokens with reasons and masked names, lower it on OOM
DELTA_PART = 512  # tokens per checkpointed part of the linear attention kernel, a multiple of its 64-token chunks. Lower saves memory, costs speed.
EPOCHS = 3
GRAD_ACCUM = 8  # 8 conversations per update
LR = 2e-4
WARMUP_STEPS = 20
LORA_R = 32
NOISE_RATE = 0.5  # share of conversations that see ASR transcripts instead of clean text
HISTORY_CALLS = True  # also learn the tool calls of history turns: 3,661 more gold calls on top of 3,541 final ones
WHY = False  # True: every learnt step starts with a short written reason, from rationales.json (made by rationales/gen_rationales.py) attached as an input
MASK_RATE = 0.3  # share of conversations whose apps get made-up names, so the model reads schemas: 65% of public samples use an app training never saw. 0 turns it off
MAX_HOURS = 9.5  # stop and save before Kaggle's 12 hour limit
MIN_FREE_GB = 3  # save and stop when host RAM available drops below this: Kaggle kills the whole session at 0
SAVE_STEPS = 50  # updates between checkpoints, a full run has 459
RESUME = False  # True: continue from the newest checkpoint in an attached output of this notebook
SEED = 42

_checkpoints = sorted(glob.glob("/kaggle/input/**/ckpt/checkpoint-*", recursive=True), key=lambda p: int(p.rsplit("-", 1)[1]))
CHECKPOINT = _checkpoints[-1] if RESUME else None
ROOT = Path(glob.glob("/kaggle/input/**/train/train.jsonl", recursive=True)[0]).parents[1]
_noisy = glob.glob("/kaggle/input/**/asr_base_train.json", recursive=True)
assert _noisy, "Add Input the output of 01c_asr_gipformer_ft.ipynb (or 01b): the LLM must train on ASR text"
NOISY_ASR = _noisy[0]
_why = glob.glob("/kaggle/input/**/rationales.json", recursive=True)
assert _why or not WHY, "WHY = True needs rationales.json attached as an input"
WHY_JSON = _why[0] if WHY else None

Checks the data path, the scorer and the gold replay before any GPU time is spent.

In [ ]:
from settings import *
!python lifemate.py "{ROOT}"
assert _exit_code == 0, "this step failed, see the log above"

In [ ]:
%%writefile train_llm.py
"""LoRA fine-tune of the LLM on gold tool trajectories. One process, the model sharded across both T4s: a full copy per GPU runs out of memory."""
import json
import os
import resource
import random
import re
import sys
import time
import unicodedata

import torch
import torch.nn.functional as F
import transformers.integrations.sdpa_attention as sdpa
import transformers.models.qwen3_5.modeling_qwen3_5 as qwen
from peft import LoraConfig, get_peft_model
from torch.utils.checkpoint import checkpoint
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments, set_seed

import lifemate as lm
from settings import *

sys.path.insert(0, os.path.dirname(NOISY_ASR))
from vi_itn import itn  # the ASR output's own ITN, so clean text gets the same digits as transcripts

START = time.time()


def free_gb():  # the whole machine, so it also counts the notebook kernel, not only this process
    return int(next(l for l in open("/proc/meminfo") if l.startswith("MemAvailable")).split()[1]) / 2**20


set_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
rows = lm.load_split(ROOT, "train")
catalog = lm.load_catalog(ROOT, "train")
rows = rows if FINAL else lm.split_rows(rows)[0]
noisy = {k: re.sub(r"[\[\]]", "", v) for k, v in json.load(open(NOISY_ASR, encoding="utf-8")).items()}  # 03c removes the tagger's marks, so training must too
whys = json.load(open(WHY_JSON, encoding="utf-8")) if WHY else {}
assert all(x["id"] in whys for x in rows) or not WHY, "rationales.json misses samples, re-run gen_rationales.py"
missing = sum(k not in noisy for x in rows for k in lm.gold_texts(x))
assert not missing, f"{missing} training clips have no ASR text, was this output a smoke run of 01?"


def old_style_tone(word):  # asr_common.norm from the 01 notebooks, so clean text takes the transcripts' form
    d = unicodedata.normalize("NFD", word)
    tones = [c for c in d if c in "̣̀́̃̉"]
    if len(tones) != 1:
        return word
    base = unicodedata.normalize("NFC", "".join(c for c in d if c not in tones))
    if base.endswith(("oa", "oe")) or (base.endswith("uy") and not base.endswith("quy")):
        return unicodedata.normalize("NFC", base[:-2] + base[-2] + tones[0] + base[-1])
    return word


def asr_style(s):
    """Test input has no capitals or punctuation, and its numbers come from ITN: clean text goes through the same norm and ITN as transcripts."""
    s = re.sub(r"[^\w\s]|_", " ", unicodedata.normalize("NFC", s).lower())
    return itn(" ".join(old_style_tone(w) for w in s.split()))


def conversation(x):
    texts = lm.gold_texts(x)
    texts = {k: noisy[k] for k in texts} if random.random() < NOISE_RATE else {k: asr_style(v) for k, v in texts.items()}  # real ASR errors, or the clean words in transcript form
    msgs = lm.build_messages(x, texts, catalog, random)
    first = len(msgs)
    msgs += lm.gold_steps(x)
    labelled = lm.targets(msgs, first, HISTORY_CALLS)
    if WHY:  # the reason goes first in the same JSON, so the model writes it before deciding. Reasons cover history calls, final calls, done
        w = whys[x["id"]][-len(labelled):]
        assert len(w) == len(labelled), x["id"]
        for i, why in zip(sorted(labelled), w):
            msgs[i] = lm.asst({"why": why} | json.loads(msgs[i]["content"]))
    if random.random() < MASK_RATE:
        msgs = lm.mask_apps(msgs, x["context"]["installed_apps"], random)
    return lm.encode(tok, msgs, labelled)


before = len(rows)
rows = [x for x in rows if len(conversation(x)["input_ids"]) <= MAX_LEN]
print(f"{len(rows)} conversations, {before - len(rows)} dropped as longer than {MAX_LEN} tokens, ASR noise: {bool(noisy)}, history calls learnt: {HISTORY_CALLS}, reasons: {WHY}, apps masked: {MASK_RATE}")
if SMOKE:  # longest conversations, so an out-of-memory error shows up now and not hours in
    rows = sorted(rows, key=lambda x: -len(conversation(x)["input_ids"]))[: SMOKE_STEPS * GRAD_ACCUM * 2]


class Conversations(torch.utils.data.Dataset):
    def __len__(self):
        return len(rows)

    def __getitem__(self, i):
        return conversation(rows[i])


def first_not_finite(model, inputs, keep):
    """Rerun a failed conversation with a hook on every module: inner modules finish first, so the first hit is where it starts."""
    names, hits = {m: n for n, m in model.named_modules()}, []

    def check(module, args, out):
        out = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(out) and out.is_floating_point() and not torch.isfinite(out).all():
            hits.append(f"{names[module]} ({out.dtype})")

    hooks = [m.register_forward_hook(check) for m in names]
    with torch.no_grad():
        model(**inputs, logits_to_keep=keep)
    for h in hooks:
        h.remove()
    return hits[0] if hits else "none, the rerun was finite"


class TargetOnlyLoss(Trainer):
    """Run lm_head on target tokens only: Qwen3.5 has 248k logits per token, about 3 GB for a 6k token prompt."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")[0, 1:]
        keep = (labels != -100).nonzero()[:, 0]
        out = model(**inputs, logits_to_keep=keep.tolist())  # a list: accelerate would move a tensor to the first GPU, and the model ends on the second
        loss = F.cross_entropy(out.logits[0].float(), labels[keep].to(out.logits.device))
        if not torch.isfinite(loss):
            bad_weights = sum(not torch.isfinite(p).all() for p in model.parameters() if p.requires_grad)
            scaler = self.accelerator.scaler.get_scale() if self.accelerator.scaler else None
            raise RuntimeError(f"Loss is not finite at step {self.state.global_step}, {inputs['input_ids'].shape[1]} tokens. Trainable weights not finite: {bad_weights}. GradScaler scale: {scaler}. First module not finite: {first_not_finite(model, inputs, keep.tolist())}. If this is fp16 on T4: switch MODEL_ID to the fallback in settings.")
        return (loss, out) if return_outputs else loss


class StopBeforeKaggleKills(TrainerCallback):
    """Kaggle discards a run at 12 h. Stop in time to merge and save, with a checkpoint at this step to resume from."""

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - START > MAX_HOURS * 3600 or free_gb() < MIN_FREE_GB:
            print(f"stopping at update {state.global_step} with a checkpoint: {time.time() - START:.0f} s, {free_gb():.1f} GB host RAM free. Resume with RESUME = True", flush=True)
            control.should_training_stop = control.should_save = True

    def on_log(self, args, state, control, **kwargs):  # a run killed at update 208 left no traceback: peak host RAM growing towards Kaggle's limit would show here
        print(f"update {state.global_step}, peak host RAM {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GB, free {free_gb():.1f} GB, peak GB per GPU", [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())], flush=True)



delta_rule = qwen.torch_chunk_gated_delta_rule


def delta_rule_in_parts(query, key, value, g, beta, initial_state=None, **kwargs):
    """The kernel of the 24 linear attention layers: whole, it keeps about 5 GB per layer for backward on the longest conversation. It already carries its state between calls, so run it in checkpointed parts."""
    kwargs["output_final_state"] = True

    def part(q, k, v, g, beta, state):
        return delta_rule(q, k, v, g=g, beta=beta, initial_state=state, **kwargs)

    outs, state = [], initial_state
    for start in range(0, query.shape[1], DELTA_PART):
        pieces = (x[:, start : start + DELTA_PART] for x in (query, key, value, g, beta))
        out, state = checkpoint(part, *pieces, state, use_reentrant=False, preserve_rng_state=False)
        outs.append(out)
    return torch.cat(outs, 1), state


assert DELTA_PART % 64 == 0
_x = [torch.randn(1, DELTA_PART + 100, 2, 8) for _ in range(3)]
_gates = dict(g=-torch.rand(1, DELTA_PART + 100, 2), beta=torch.rand(1, DELTA_PART + 100, 2), use_qk_l2norm_in_kernel=True)
assert torch.allclose(delta_rule_in_parts(*_x, **_gates)[0], delta_rule(*_x, **_gates)[0], rtol=1e-3, atol=1e-4), "parts differ from the whole kernel"
qwen.torch_chunk_gated_delta_rule = delta_rule_in_parts
sdpa.use_gqa_in_sdpa = lambda *args: False  # a T4 has no grouped-key attention kernel: it would fall back to math, 4 float32 tensors of 16 x T^2, about 2 GB each at 5,900 tokens
torch.backends.cuda.enable_math_sdp(False)  # so a fallback stops at the first step with "No available kernel" and not out of memory

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto" if torch.cuda.is_available() else None, attn_implementation="sdpa")
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM"))  # adapters stay fp32, so the fp16 GradScaler works
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="ckpt",
    per_device_train_batch_size=1,  # one conversation is several thousand tokens
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    max_steps=SMOKE_STEPS if SMOKE else -1,
    learning_rate=LR,
    optim="adamw_torch",  # not the fused default: it skips an overflowed step only if found_inf == 1, and GradScaler sums devices, so two GPUs overflowing together give 2 and NaN goes into every weight
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=1.0,
    fp16=FP16,
    use_cpu=not torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    label_names=["labels"],
    logging_steps=10,
    disable_tqdm=True,  # a progress bar per update flooded the kernel's output, which stopped flushing for an hour
    save_strategy="steps",
    save_steps=10 if SMOKE else SAVE_STEPS,  # the smoke proves that saving works
    save_total_limit=1,
    report_to="none",
    seed=SEED,
)
trainer = TargetOnlyLoss(
    model=model,
    args=args,
    train_dataset=Conversations(),
    data_collator=lambda batch: {k: torch.tensor([batch[0][k]]) for k in batch[0]},
    callbacks=[StopBeforeKaggleKills()],
)
try:
    trainer.train(resume_from_checkpoint=CHECKPOINT)
finally:
    print("peak GB per GPU:", [round(torch.cuda.max_memory_allocated(i) / 2**30, 1) for i in range(torch.cuda.device_count())], "GradScaler scale:", trainer.accelerator.scaler.get_scale() if trainer.accelerator.scaler else None)
trainer.save_model("llm_lora")

In [ ]:
# train.log survives in the output even when the notebook's own log stops flushing. Exit code 137 means killed, usually out of host RAM
!bash -o pipefail -c "TOKENIZERS_PARALLELISM=false PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -u train_llm.py 2>&1 | tee train.log"
assert _exit_code == 0, f"this step failed with exit code {_exit_code}, see the log above or train.log"

Merge the LoRA into the text-only model. It loads on CPU, the GPUs are done.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
model = PeftModel.from_pretrained(base, "llm_lora").merge_and_unload()
params = sum(p.numel() for p in model.parameters())
assert params < 5e9, f"over the LLM limit: {params:,}"
print(f"LLM parameters: {params:,}")
model.save_pretrained("llm_model")
AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained("llm_model")
!ls -la llm_model